In [ ]:
What is Self-Instruct?

Self-Instruct is a method to generate instruction-response pairs automatically from a pretrained LLM.

Instead of manually creating labeled datasets for instruction fine-tuning, the model generates its own tasks and outputs.

These generated pairs are then used to fine-tune a smaller or base model, improving instruction-following capabilities.

In [ ]:
How It Works

Start with a base LLM (e.g., GPT-3, GPT-4, or LLaMA).

Provide a few seed instructions to the model.

Example: “Translate English to French”, “Summarize this paragraph”, “Classify sentiment”.

Model generates new instruction-response pairs by rephrasing or creating similar tasks.

Example:

Input: “Summarize the text” → Model outputs a new instruction: “Give a short summary of this article in 2 sentences.”

Response is generated automatically.

Collect thousands of these synthetic pairs → this is your augmented dataset.

In [ ]:
Why it is useful?

| Benefit                     | Explanation                                                                           |
| --------------------------- | ------------------------------------------------------------------------------------- |
| **Scalable**                | Generates tens of thousands of instruction-response examples without manual labeling. |
| **Diverse instructions**    | The model can create new variations of instructions not seen in the original dataset. |
| **Improved generalization** | Fine-tuned models on self-instruct data can handle **zero-shot tasks** better.        |
| **Efficient**               | Reduces reliance on large manually labeled instruction datasets (like FLAN, Alpaca).  |


In [ ]:
Base LLM (pretrained)
        │
        ▼
Seed Instructions + Prompts
        │
        ▼
LLM generates synthetic instruction-response pairs
        │
        ▼
Augmented dataset for instruction fine-tuning
        │
        ▼
Fine-tuned smaller model that follows instructions


In [4]:
from transformers import pipeline

# Base LLM pipeline
generator = pipeline("text-generation", model="gpt2")

# Seed instruction
seed = "Summarize this paragraph in one sentence: Artificial intelligence is transforming the world."

# Generate synthetic instruction-response
synthetic_output = generator(f"Instruction: {seed}\nResponse:", max_length=100, num_return_sequences=3)

for i, out in enumerate(synthetic_output):
    print(f"Example {i+1}: {out['generated_text']}")


Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example 1: Instruction: Summarize this paragraph in one sentence: Artificial intelligence is transforming the world.
Response: It is becoming increasingly difficult to imagine any future that will not lead to the realization of this human-machine hybridization. (One important aspect of this is that there is a growing awareness that AI is a real possibility.)
Some of the examples I have pointed out are:
The last few years of the 20th century have witnessed a massive shift in the way we think about our world in many important ways.
In the US, for instance, we have been able to build a huge number of mass surveillance satellites, which are designed for military purposes only, and which are in use for intelligence gathering.
In the UK, we have been able to build huge amounts of surveillance satellites that are not only for surveillance purposes but for intelligence gathering.
In the US, we have been able to build massive amounts of surveillance satellites that are not only for surveillance

In [5]:
from transformers import pipeline

#  Load GPT-2 text-generation pipeline
generator = pipeline("text-generation", model="gpt2")

# Seed instruction (Question generation)
seed = "Explain why climate change is a pressing global issue in one sentence."

# Generate synthetic instruction-response
synthetic_output = generator(f"Instruction: {seed}\nResponse:", max_length=20, num_return_sequences=2)

# Display generated examples
for i, out in enumerate(synthetic_output):
    print(f"Example {i+1}:\n{out['generated_text']}\n")


Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Example 1:
Instruction: Explain why climate change is a pressing global issue in one sentence.
Response: "The most pressing global issue is the rise of extreme precipitation over the past 10 years."
Response: "The most urgent global issue is climate change. The most urgent global issue is the rise of extreme precipitation over the past 10 years."

Example 2:
Instruction: Explain why climate change is a pressing global issue in one sentence.
Response: It is not.
Climate change is a global problem—not a global problem of natural variability. It is a global problem of human-made global changes.
It is not. The United States is responsible for three out of four of the climate change attributable to human-made global changes.
A third of all of the climate change attributable to human-made global changes is caused by human activity.
In other words, if you ask the question "What is the relationship between human activity and climate change?" you are likely to get an answer that is not exactly 

In [6]:
# sample instructions 

# ------------------------------
# Self-Instruct Data Generation
# ------------------------------
# !pip install -q transformers datasets

import torch
from transformers import pipeline
from datasets import Dataset
import pandas as pd

# Load GPT-2 generation pipeline
device = 0 if torch.cuda.is_available() else -1
generator = pipeline("text-generation", model="gpt2", device=device)

# Define seed instructions
seed_instructions = [
    "Summarize this paragraph in one sentence: Artificial intelligence is transforming the world.",
    "Explain why climate change is a pressing global issue in one sentence.",
    "Describe the benefits of exercise for mental health.",
    "Provide a tip for improving productivity at work."
]

# Generate synthetic instruction-response pairs
synthetic_data = []
for seed in seed_instructions:
    outputs = generator(f"Instruction: {seed}\nResponse:", 
                        max_length=50, 
                        num_return_sequences=5)  # 5 variations per seed
    for out in outputs:
        text = out['generated_text']
        # Split instruction vs response
        if "Response:" in text:
            instr, resp = text.split("Response:", 1)
            synthetic_data.append({"instruction": instr.replace("Instruction:", "").strip(),
                                   "response": resp.strip()})

# Convert to Hugging Face Dataset
df = pd.DataFrame(synthetic_data)
dataset = Dataset.from_pandas(df)

# Preview
print("Number of generated examples:", len(dataset))
print(dataset[:5])


Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformer

Number of generated examples: 20
{'instruction': ['Summarize this paragraph in one sentence: Artificial intelligence is transforming the world.', 'Summarize this paragraph in one sentence: Artificial intelligence is transforming the world.', 'Summarize this paragraph in one sentence: Artificial intelligence is transforming the world.', 'Summarize this paragraph in one sentence: Artificial intelligence is transforming the world.', 'Summarize this paragraph in one sentence: Artificial intelligence is transforming the world.'], 'response': ['"We don\'t know whether artificial intelligence is in use in the future, or whether it will be used until the end of the next century."\nResponse: "I don\'t know what an artificial intelligence is anyway."\nResponse: "I don\'t like the term, but I don\'t think the current term is wrong. It\'s very broad and broad, and it\'s quite simple."\nResponse: "There are many types of AI that could become real. I don\'t think there are any of them that are as in

In [ ]:
# limitations of instruction tuning

In [ ]:
1 Dependence on Quality of Training Data

Instruction fine-tuning relies heavily on high-quality instruction-response pairs.
If the dataset contains ambiguous or incorrect instructions, the model may learn wrong behaviors.
Generating large, high-quality datasets manually is time-consuming and expensive.

2 Limited Generalization to Unseen Tasks

Even after instruction tuning, models may struggle with tasks very different from the fine-tuning data.
Example: A model fine-tuned on summarization and translation may not do well on code generation unless those instructions were included.

3 Overfitting to Instruction Format

The model may learn to follow a rigid instruction style.
Slight variations in prompt wording can lead to unexpected or poor outputs.

4 Increased Training Cost

Instruction tuning requires additional compute resources on top of pretraining.
Large models (like GPT-3 or LLaMA) may require multiple GPUs or TPUs.

5 Safety & Bias Risks

Fine-tuned models can amplify biases in the training instructions.
Instructions may inadvertently encourage harmful or biased outputs if not carefully curated.

6 Model Size Limitation

Instruction fine-tuning is often more effective for larger models.

Small models may not fully benefit from instruction tuning and may still produce incoherent outputs.

# RLHF (Reinforcement Learning from Human Feedback)

In [ ]:
1. What is RLHF?

RLHF is a technique to align large language models (LLMs) with human preferences.
Instead of only training on next-token prediction, the model learns to produce outputs that humans consider helpful, safe, or high-quality.

2. Why RLHF is Needed

Even instruction-fine-tuned models have limitations:
    May generate undesirable outputs
    Factually incorrect statements
    Toxic or biased content
    Overly verbose or irrelevant responses
Cannot perfectly follow human preferences
    Standard fine-tuning optimizes likelihood of data, not human satisfaction
Hard to encode complex human objectives in simple loss functions

Motivation: RLHF lets the model learn from human feedback directly, producing outputs humans prefer.

3 How RLHF Works (High-Level)

3.1 Pretraining / Fine-tuning
Train a language model on large text corpus → general knowledge.

3.2 Collect Human Feedback
Humans rank or score multiple outputs for the same prompt.
Example: “Which of these responses is more helpful or accurate?”

3.3 Train Reward Model (RM)
Model predicts the human preference score for outputs.

3.4 Policy Optimization
Use reinforcement learning (e.g., PPO) to update the language model so it maximizes the reward predicted by the RM.

In [ ]:
| Benefit                          | Explanation                                                 |
| -------------------------------- | ----------------------------------------------------------- |
| **Alignment with human values**  | Produces responses that humans actually want.               |
| **Reduces harmful outputs**      | Helps avoid toxic, biased, or unsafe content.               |
| **Improved usefulness**          | Outputs are more helpful, concise, and relevant.            |
| **Better instruction following** | Goes beyond likelihood training to match human preferences. |


In [2]:
#!pip install transformers torch datasets trl -q
!pip install -q transformers torch datasets trl


In [2]:
# -------------------------------
# RLHF Supervised Demo
# -------------------------------
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset
import torch

# Load GPT-2 small and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 does not have pad_token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Synthetic instruction-response dataset
data = [
    {"instruction": "Translate to French: I love AI", "response": "J'adore l'IA"},
    {"instruction": "Translate to French: Machine learning is fun", "response": "L'apprentissage automatique est amusant"},
    {"instruction": "Translate to French: Hello world", "response": "Bonjour le monde"}
]

# Combine instruction + response as input for causal LM
def combine_input(example):
    # Format: Instruction + Response
    text = f"Instruction: {example['instruction']}\nResponse: {example['response']}"
    return {"text": text}

dataset = Dataset.from_list(data)
dataset = dataset.map(combine_input)

# Tokenize
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()  # for causal LM
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Define TrainingArguments
training_args = TrainingArguments(
    output_dir="./rlhf_demo",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_steps=5,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to=[],          # disables wandb/other logging
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Train
trainer.train()


prompt = "Instruction: Translate to French: AI is amazing\nResponse:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)  # <-- move to device
outputs = model.generate(**inputs, max_length=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))




Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Step,Training Loss
1,4.681700
2,2.814100
3,1.980600


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Translate to French: AI is amazing
Response: I'm not sure what I'm going to say.


In [3]:
# -------------------------------
# Instruction-following demo with FLAN-T5
# -------------------------------
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from datasets import Dataset
import torch

# Load FLAN-T5 small and tokenizer
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Synthetic instruction-response dataset (tiny demo)
data = [
    {"instruction": "Translate to French: I love AI", "response": "J'adore l'IA"},
    {"instruction": "Translate to French: Machine learning is fun", "response": "L'apprentissage automatique est amusant"},
    {"instruction": "Translate to French: Hello world", "response": "Bonjour le monde"},
    {"instruction": "Summarize: Artificial intelligence is transforming the world", "response": "AI is changing the world"},
    {"instruction": "Explain: What is supervised learning?", "response": "Learning with labeled data"},
    {"instruction": "Translate to Spanish: I love Python", "response": "Me encanta Python"},
]

# Combine instruction + response as input for Seq2Seq LM
def combine_input(example):
    return {"text": f"Instruction: {example['instruction']}\nResponse: {example['response']}"}

dataset = Dataset.from_list(data)
dataset = dataset.map(combine_input)

# Tokenize
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()  # labels for seq2seq LM
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Define TrainingArguments
training_args = TrainingArguments(
    output_dir="./flan_t5_demo",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_steps=5,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to=[],  # disables wandb
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

# Train
trainer.train()

# Test instruction-following
prompts = [
    "Translate to French: AI is amazing",
    "Summarize: Machine learning helps businesses",
    "Translate to Spanish: I love data science"
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_length=64)
    print(f"Prompt: {prompt}")
    print(f"Response: {tokenizer.decode(outputs[0], skip_special_tokens=True)}\n")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

/home/monsley/anaconda3/envs/E3/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
1,32.891500
2,32.210500
3,29.775700
4,29.732500
5,30.691900
6,30.613400


/home/monsley/anaconda3/envs/E3/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Prompt: Translate to French: AI is amazing
Response: AI est en uvre

Prompt: Summarize: Machine learning helps businesses
Response: Inventors learn how to learn to use technology

Prompt: Translate to Spanish: I love data science
Response: Je encuentra en la sabida satélite de los datos



In [ ]:
# EXAMPLE RLHF 

In [ ]:
# Environment
pip install torch transformers datasets accelerate
pip install trl peft

In [1]:
# STAGE 1 MODEL

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("Model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

C:\Users\fedex\anaconda3\envs\ABMLLM\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fedex\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


In [2]:
prompt = "Machine learning is"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=30
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Machine learning is a very powerful tool for learning about the world around us. It's a tool that can help us understand the world around us, and it's a


In [ ]:
# Stage 2 — Create Preference Data

In [3]:
from datasets import Dataset

# Example prompts
prompts = [
    "What is machine learning?",
    "Explain neural networks to a beginner.",
    "What is the difference between AI and ML?",
    "Why is data preprocessing important?"
]

# Response A and Response B for each prompt
responses_a = [
    "Machine learning is a branch of AI where computers learn patterns from data to make predictions or decisions.",
    "A neural network is a computing model inspired by the brain. It contains connected layers of artificial neurons that learn patterns from data.",
    "Artificial intelligence is the broader field of creating systems that perform tasks requiring intelligence. Machine learning is a subset of AI that learns from data.",
    "Data preprocessing cleans and transforms raw data so that machine learning algorithms can learn more effectively."
]

responses_b = [
    "Machine learning is when a computer becomes intelligent by itself.",
    "Neural networks are basically computers that think like humans.",
    "AI and ML are exactly the same thing.",
    "Data preprocessing is not important because machine learning models can handle any data."
]

# Human preference
# 1 = Response A is preferred
# 0 = Response B is preferred
preference = [1, 1, 1, 1]

# Create a Hugging Face Dataset
dataset = Dataset.from_dict({
    "prompt": prompts,
    "response_a": responses_a,
    "response_b": responses_b,
    "preference": preference
})

print(dataset)

Dataset({
    features: ['prompt', 'response_a', 'response_b', 'preference'],
    num_rows: 4
})


In [ ]:
| Prompt                  | Response A | Response B     | Human preference |
| ----------------------- | ---------- | -------------- | ---------------- |
| What is ML?             | Clear      | Vague          | A                |
| Explain neural networks | Detailed   | Oversimplified | A                |
| AI vs ML                | Correct    | Incorrect      | A                |
| Why preprocessing?      | Correct    | Incorrect      | A                |


In [ ]:
# Stage 3 — Build the Reward Model

In [4]:
# 1. Load a model with a reward/classification head

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_NAME = "distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load pretrained model with a single output score
reward_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1
)

print("Reward model loaded!")


# Why num_labels=1?
# Because we don't want:

# Class 0
# Class 1

# We want a continuous reward score:
# Response A → 0.82
# Response B → 0.31

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\fedex\anaconda3\envs\ABMLLM\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fedex\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Reward model loaded!


In [ ]:
## 2. Prepare the preference pairs

In [ ]:
# Our dataset looks like:

# Prompt
#    +
# Response A
#    +
# Response B
#    +
# Human preference

# We need to combine the prompt and each response.

In [5]:
def create_pair(example):

    # Combine the prompt with Response A
    text_a = example["prompt"] + "\n" + example["response_a"]

    # Combine the prompt with Response B
    text_b = example["prompt"] + "\n" + example["response_b"]

    return {
        "text_a": text_a,
        "text_b": text_b,
        "preference": example["preference"]
    }
preference_dataset = dataset.map(create_pair)

print(preference_dataset[0])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

{'prompt': 'What is machine learning?', 'response_a': 'Machine learning is a branch of AI where computers learn patterns from data to make predictions or decisions.', 'response_b': 'Machine learning is when a computer becomes intelligent by itself.', 'preference': 1, 'text_a': 'What is machine learning?\nMachine learning is a branch of AI where computers learn patterns from data to make predictions or decisions.', 'text_b': 'What is machine learning?\nMachine learning is when a computer becomes intelligent by itself.'}


In [ ]:
## 3. Tokenize the responses
# The neural network doesn't understand strings directly.

In [6]:
def tokenize_pairs(example):

    # Tokenize Prompt + Response A
    tokens_a = tokenizer(
        example["text_a"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # Tokenize Prompt + Response B
    tokens_b = tokenizer(
        example["text_b"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    return {
        "input_ids_a": tokens_a["input_ids"],
        "attention_mask_a": tokens_a["attention_mask"],
        "input_ids_b": tokens_b["input_ids"],
        "attention_mask_b": tokens_b["attention_mask"],
        "preference": example["preference"]
    }


tokenized_dataset = preference_dataset.map(
    tokenize_pairs
)

print(tokenized_dataset)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'response_a', 'response_b', 'preference', 'text_a', 'text_b', 'input_ids_a', 'attention_mask_a', 'input_ids_b', 'attention_mask_b'],
    num_rows: 4
})


In [ ]:
## The Reward Model Architecture

              Prompt
                │
        ┌───────┴───────┐
        ▼               ▼
   Response A       Response B
        │               │
        ▼               ▼
   DistilBERT        DistilBERT
        │               │
        ▼               ▼
    Reward A        Reward B
        │               │
        └───────┬───────┘
                ▼
          Compare Scores
                │
                ▼
       Human Preference

## The model needs to learn to produce this relationship.

In [ ]:
## 5. The Reward Loss 

This is the important mathematical part.

Suppose:

$$ r_A = \text{Reward}(A) $$

and

$$ r_B = \text{Reward}(B) $$

We want:

$$ r_A > r_B $$

when humans prefer A.

A commonly used preference loss is:

$$ L=-\log(\sigma(r_A-r_B)) $$

where:

$$ \sigma(x)=\frac{1}{1+e^{-x}} $$

So if:

rA = 0.9
rB = 0.2

then:

rA - rB = 0.7

and the model gets a relatively good loss.

But if:

rA = 0.2
rB = 0.9

the loss becomes large.

In [ ]:
## The reward model is punished whenever it gives the preferred response a lower score.

In [ ]:
# 6. Implement the Reward Model

In [7]:
import torch
import torch.nn as nn


class RewardModel(nn.Module):

    def __init__(self, base_model):
        super().__init__()

        # Pretrained language representation model
        self.base_model = base_model

    def forward(
        self,
        input_ids_a,
        attention_mask_a,
        input_ids_b,
        attention_mask_b
    ):

        # Get reward score for Response A
        output_a = self.base_model(
            input_ids=input_ids_a,
            attention_mask=attention_mask_a
        )

        # Get reward score for Response B
        output_b = self.base_model(
            input_ids=input_ids_b,
            attention_mask=attention_mask_b
        )

        reward_a = output_a.logits.squeeze(-1)
        reward_b = output_b.logits.squeeze(-1)

        return reward_a, reward_b

In [19]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1
)

reward_model = RewardModel(base_model)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
## 7. Define the Preference Loss

In [9]:
def preference_loss(reward_a, reward_b):

    # We want reward_a > reward_b
    # when humans prefer Response A

    loss = -torch.nn.functional.logsigmoid(
        reward_a - reward_b
    ).mean()

    return loss

In [ ]:
# 8. What Are We Training?

Initially, the reward model might produce:

Prompt: Explain machine learning
        Response A → 0.31
        Response B → 0.72
But we (humans) said:
A > B
So the model is wrong.

Training pushes it toward:
    Response A → 0.85
    Response B → 0.27
Now:
A > B
The reward model has learned something about human preference.

In [ ]:
# First three stages

        LARGE CORPUS
             │
             ▼
       PRETRAINING
             │
             ▼
          Base LLM
             │
             ▼
            SFT
             │
             ▼
    Instruction Model
             │
       generates
       responses
             │
             ▼
   ┌───────────────────┐
   │ Human Preferences │
   │                   │
   │ A > B             │
   └─────────┬─────────┘
             │
             ▼
       REWARD MODEL
             │
       A → 0.85
       B → 0.27
             │
             ▼
       Next: PPO / RL

In [ ]:
# Stage 4 — Reinforcement Learning

                    Prompt
                      │
                      ▼
                ┌───────────┐
                │   GPT-2   │
                │  Policy   │
                └─────┬─────┘
                      │
                Generate response
                      │
                      ▼
                ┌───────────┐
                │  Reward   │
                │   Model   │
                └─────┬─────┘
                      │
                  Reward
                      │
                      ▼
                ┌───────────┐
                │    PPO    │
                └─────┬─────┘
                      │
                      ▼
              Updated GPT-2

In [ ]:
## The Reward Model evaluates the response, and PPO uses that reward to update the LLM

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

policy_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

# GPT-2 does not have a padding token by default
tokenizer.pad_token = tokenizer.eos_token

policy_model.config.pad_token_id = tokenizer.pad_token_id

In [12]:
# 1. let's generate a response
prompt = "Machine learning is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

with torch.no_grad():

    outputs = policy_model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Machine learning is the study of human relationships with the environment and the way the people live their lives. In the context of learning, social science research is often defined as


In [ ]:
# 2. The LLM is the Policy

This is an important RL concept.
We call the GPT-2 model:
    Policy
because it determines what action to take.

                        Prompt
                          ↓
                        Policy (GPT-2)
                          ↓
                        Token 1
                          ↓
                        Token 2
                          ↓
                        Token 3
                          ↓
                        ...
                          ↓
                        Response

Mathematically, the policy is:

$$ \pi_\theta(y|x) $$

where:

\(x\) = prompt
\(y\) = generated response
\(\theta\) = parameters of GPT-2

In [ ]:
# 3. Send the Response to the Reward Model

Remember Reward Model, It takes:
    Prompt + Response
and produces:
    Reward score



In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# -----------------------------------
# 1. Reward Model Configuration
# -----------------------------------

MODEL_NAME = "distilbert-base-uncased"

# -----------------------------------
# 2. Reward Tokenizer
# -----------------------------------

reward_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# -----------------------------------
# 3. Base Reward Model
# -----------------------------------

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1
)

# -----------------------------------
# 4. Wrap it as our Reward Model
# -----------------------------------

reward_model = RewardModel(base_model)

print("Reward tokenizer loaded!")
print("Reward model loaded!")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Reward tokenizer loaded!
Reward model loaded!


In [24]:
def get_reward(prompt, response):

    # Combine prompt and response
    text = prompt + "\n" + response

    # Convert text into tokens
    inputs = reward_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # Get reward score
    with torch.no_grad():

        output = reward_model.base_model(
            **inputs
        )

    # Convert model output to a single number
    reward = output.logits.squeeze().item()

    return reward

In [25]:
prompt = "What is machine learning?"

response = "Machine learning is a method where computers learn patterns from data."

reward = get_reward(prompt, response)

print("Reward:", reward)

Reward: -0.015972593799233437


In [ ]:
                    PREFERENCE DATA
                          │
              ┌───────────┴───────────┐
              ↓                       ↓
        Response A              Response B
        (preferred)              (rejected)
              │                       │
              └───────────┬───────────┘
                          ↓
                    Reward Model
                          │
                    TRAINING
                          ↓
              A gets higher reward
              B gets lower reward
                          │
                          ↓
                       PPO/RL

In [ ]:
# Step 3A — Train the Reward Model

Prompt
  │
  ├── Response A ──→ Reward Model ──→ rA
  │
  └── Response B ──→ Reward Model ──→ rB

We want:

             rA > rB

Prompt:
"What is machine learning?"

A:
"Machine learning allows computers to learn patterns
from data and make predictions."

B:
"Machine learning is when computers become intelligent."

Human preference:
A > B

In [26]:
# Lets prepare the training data

import torch
from torch.utils.data import Dataset, DataLoader


class PreferenceDataset(Dataset):

    def __init__(self, prompts, responses_a, responses_b):

        self.prompts = prompts
        self.responses_a = responses_a
        self.responses_b = responses_b

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):

        return {
            "prompt": self.prompts[idx],
            "response_a": self.responses_a[idx],
            "response_b": self.responses_b[idx]
        }


dataset = PreferenceDataset(
    prompts,
    responses_a,
    responses_b
)

print("Number of examples:", len(dataset))

Number of examples: 4


In [ ]:
# 2. Create the optimizer

We want to update the reward model's parameters.


In [27]:
optimizer = torch.optim.AdamW(
    reward_model.parameters(),
    lr=2e-5
)

In [28]:
## 3. Define the preference loss

import torch.nn.functional as F


def preference_loss(reward_a, reward_b):

    loss = -F.logsigmoid(
        reward_a - reward_b
    ).mean()

    return loss

In [29]:
## 4. TRAINING THE REWARD MODELLLL

num_epochs = 10

for epoch in range(num_epochs):

    reward_model.train()

    total_loss = 0

    for example in dataset:

        # --------------------------------
        # Prepare Response A
        # --------------------------------

        text_a = (
            example["prompt"]
            + "\n"
            + example["response_a"]
        )

        inputs_a = reward_tokenizer(
            text_a,
            return_tensors="pt",
            truncation=True,
            max_length=128
        )

        # --------------------------------
        # Prepare Response B
        # --------------------------------

        text_b = (
            example["prompt"]
            + "\n"
            + example["response_b"]
        )

        inputs_b = reward_tokenizer(
            text_b,
            return_tensors="pt",
            truncation=True,
            max_length=128
        )

        # --------------------------------
        # Get reward scores
        # --------------------------------

        reward_a, reward_b = reward_model(
            input_ids_a=inputs_a["input_ids"],
            attention_mask_a=inputs_a["attention_mask"],

            input_ids_b=inputs_b["input_ids"],
            attention_mask_b=inputs_b["attention_mask"]
        )

        # --------------------------------
        # Calculate preference loss
        # --------------------------------

        loss = preference_loss(
            reward_a,
            reward_b
        )

        # --------------------------------
        # Backpropagation
        # --------------------------------

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{num_epochs}, "
        f"Loss: {total_loss / len(dataset):.4f}"
    )

Epoch 1/10, Loss: 0.6858
Epoch 2/10, Loss: 0.6417
Epoch 3/10, Loss: 0.5807
Epoch 4/10, Loss: 0.5250
Epoch 5/10, Loss: 0.3893
Epoch 6/10, Loss: 0.2632
Epoch 7/10, Loss: 0.1609
Epoch 8/10, Loss: 0.1017
Epoch 9/10, Loss: 0.0686
Epoch 10/10, Loss: 0.0461


In [30]:
# 5.Testing the reward models


prompt = "What is machine learning?"

response_a = (
    "Machine learning is a method where computers "
    "learn patterns from data to make predictions."
)

response_b = (
    "Machine learning is when computers become "
    "intelligent by themselves."
)

reward_a = get_reward(prompt, response_a)
reward_b = get_reward(prompt, response_b)

print("Reward A:", reward_a)
print("Reward B:", reward_b)

Reward A: 1.7792487144470215
Reward B: -1.2538573741912842


In [ ]:
# We now have a tiny preference reward model

                PREFERENCE DATA
                       │
              Prompt + A + B
                       │
                       ↓
              ┌─────────────────┐
              │  Reward Model   │
              │   DistilBERT    │
              └────────┬────────┘
                       │
              ┌────────┴────────┐
              ↓                 ↓
           Reward A          Reward B
              │                 │
              └───────┬─────────┘
                      ↓
                  A should > B
                      │
                      ↓
               Preference Loss
                      │
                      ↓
                 Backpropagation
                      │
                      ↓
              Update Reward Model

In [ ]:
# Coming back to GPT!

In [ ]:
                    GPT-2
                      │
                   Prompt
                      ↓
                Generate response
                      ↓
               Generated Response
                      ↓
               Reward Tokenizer
                      ↓
                Reward Model
                      ↓
                 Reward Score
                      ↓
                PPO / RL Update
                      ↓
                 GPT-2 improves

In [ ]:
# Stage 4: PPO / Reinforcement Learning.

In [ ]:
## Stage 4.1 — Understand the PPO setup

| Component                 | Role                                    |
| ------------------------- | --------------------------------------- |
| **GPT-2 Policy Model**    | Generates responses                     |
| **GPT-2 Reference Model** | Keeps the policy from changing too much |
| **Reward Model**          | Judges generated responses              |
| **Value Model / Critic**  | Estimates expected future reward        |
| **PPO**                   | Updates the policy                      |


In [ ]:
                    Prompt
                      │
                      ▼
                ┌───────────┐
                │   GPT-2   │
                │  POLICY   │
                └─────┬─────┘
                      │
                      ▼
                Generated Text
                      │
                      ▼
             ┌─────────────────┐
             │  Reward Model   │
             │   DistilBERT    │
             └────────┬────────┘
                      │
                      ▼
                 Reward = 2.4
                      │
                      ▼
                  ┌───────┐
                  │  PPO  │
                  └───┬───┘
                      │
                      ▼
             Update GPT-2 Policy

In [31]:
import transformers
import trl

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)

Transformers: 4.57.0.dev0
TRL: 0.28.0


In [32]:
import torch

print("PyTorch:", torch.__version__)

PyTorch: 2.8.0+cu129


In [33]:
# 
from trl.experimental.ppo import PPOConfig, PPOTrainer

print("PPO imported successfully!")

C:\Users\fedex\AppData\Local\Temp\ipykernel_19380\617159940.py:3: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.ppo import PPOConfig, PPOTrainer



PPO imported successfully!


In [34]:
# 2. Loading GPT-2 policy

from transformers import AutoTokenizer, AutoModelForCausalLM

POLICY_MODEL_NAME = "gpt2"

policy_tokenizer = AutoTokenizer.from_pretrained(
    POLICY_MODEL_NAME
)

policy_model = AutoModelForCausalLM.from_pretrained(
    POLICY_MODEL_NAME
)

print("Policy model loaded!")

Policy model loaded!


In [ ]:
## 3. Create the reference model
### PPO needs a frozen copy of the original policy

                 GPT-2
                  │
        ┌─────────┴─────────┐
        ↓                   ↓
   Policy Model       Reference Model
   (TRAINABLE)          (FROZEN)
        │                   │
        └─────────┬─────────┘
                  ↓
              PPO / KL

In [35]:
import copy

ref_model = copy.deepcopy(policy_model)

for param in ref_model.parameters():
    param.requires_grad = False

print("Reference model created!")

Reference model created!


In [ ]:
## 4. What about the Value Model?

PPO needs a critic/value model to estimate:
"Given this state, how much reward should I expect?"

In [ ]:
GPT-2 generates tokens

"The weather today is..."

       ↓

Value Model

       ↓

Expected future reward
       ↓
      V(s)

In [36]:
# 

import inspect
from trl.experimental.ppo import PPOTrainer, PPOConfig

print(inspect.signature(PPOTrainer))

print(inspect.signature(PPOConfig))

import trl.experimental.ppo as ppo

print(dir(ppo))

(args: trl.experimental.ppo.ppo_config.PPOConfig, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.image_processing_utils.BaseImageProcessor | transformers.feature_extraction_utils.FeatureExtractionMixin | transformers.processing_utils.ProcessorMixin, model: torch.nn.modules.module.Module, ref_model: torch.nn.modules.module.Module | None, reward_model: torch.nn.modules.module.Module, train_dataset: datasets.arrow_dataset.Dataset, value_model: torch.nn.modules.module.Module, data_collator: transformers.data.data_collator.DataCollatorWithPadding | None = None, eval_dataset: datasets.arrow_dataset.Dataset | dict[str, datasets.arrow_dataset.Dataset] | None = None, optimizers: tuple[torch.optim.optimizer.Optimizer, torch.optim.lr_scheduler.LambdaLR] = (None, None), callbacks: list[transformers.trainer_callback.TrainerCallback] | None = None, peft_config: 'PeftConfig | None' = None) -> None
(output_dir: Optional[str] = None, overwrite_output_dir: 

In [ ]:
                 PROMPT
                    │
                    ▼
              ┌──────────┐
              │  GPT-2   │
              │  POLICY  │ ◄───────────── ┐
              └────┬─────┘                │
                   │                      │
                   ▼                      │
              RESPONSE                    │
                   │                      │
                   ▼                      │
          ┌────────────────┐              │
          │ Reward Model   │              │
          │  DistilBERT    │              │
          └───────┬────────┘              │
                  │                       │
                  ▼                       │
                REWARD                    │
                  │                       │
        ┌─────────┴─────────┐             │
        │                   │             │
        ▼                   ▼             │
  Value Model        Reference GPT-2      │
        │                   │             │
        └─────────┬─────────┘             │
                  ▼                       │
                PPO ──────────────────────┘

In [ ]:
# Stage 4 — PPO
4.1 Import PPO components

In [37]:
from trl.experimental.ppo import (
    PPOConfig,
    PPOTrainer,
    AutoModelForCausalLMWithValueHead
)

print("PPO components imported!")

PPO components imported!


In [38]:
# 4.2 preparing GPT 2

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

POLICY_MODEL_NAME = "gpt2"

policy_tokenizer = AutoTokenizer.from_pretrained(
    POLICY_MODEL_NAME
)

policy_model = AutoModelForCausalLM.from_pretrained(
    POLICY_MODEL_NAME
)

# GPT-2 does not have a pad token by default
policy_tokenizer.pad_token = policy_tokenizer.eos_token

print("Policy model loaded!")

Policy model loaded!


In [39]:
# 4.3  reference model
import copy

ref_model = copy.deepcopy(policy_model)

for param in ref_model.parameters():
    param.requires_grad = False

print("Reference model created!")

Reference model created!


In [51]:
from transformers import AutoModelForSequenceClassification

value_model = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=1
)

print(type(value_model))
print("base_model_prefix:", value_model.base_model_prefix)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<class 'transformers.models.gpt2.modeling_gpt2.GPT2ForSequenceClassification'>
base_model_prefix: transformer


In [62]:
# 4.5 Prepare the Reward Model
from transformers import AutoModelForSequenceClassification

ppo_reward_model = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=1
)

print(type(ppo_reward_model))
print("base_model_prefix:",
      ppo_reward_model.base_model_prefix)

ppo_reward_model.config.pad_token_id = policy_tokenizer.eos_token_id

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<class 'transformers.models.gpt2.modeling_gpt2.GPT2ForSequenceClassification'>
base_model_prefix: transformer


In [58]:
# 4.6   PPO dataset

from datasets import Dataset

# Create dataset containing prompts
ppo_dataset = Dataset.from_dict({
    "prompt": prompts
})

# Tokenize the prompts
def tokenize_prompt(example):

    tokens = policy_tokenizer(
        example["prompt"],
        truncation=True,
        max_length=128
    )

    return tokens


ppo_dataset = ppo_dataset.map(
    tokenize_prompt,
    batched=False
)

# Keep only the columns required by PPO
ppo_dataset = ppo_dataset.remove_columns(["prompt"])

print(ppo_dataset)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 4
})


In [66]:
# 4.7 Configure PPO

ppo_config = PPOConfig(
    output_dir="./ppo_output",

    per_device_train_batch_size=1,

    learning_rate=1e-5,
    num_train_epochs=1,
    max_steps=4,

    logging_steps=1,
    report_to="none",

    num_mini_batches=1,
    num_ppo_epochs=2,

    kl_coef=0.05,
    cliprange=0.2,
    vf_coef=0.1,

    response_length=30,
    temperature=0.7,

    # IMPORTANT
    num_sample_generations=0
)

In [55]:
# 4.8 Create the PPO Trainer

In [67]:
from trl.experimental.ppo import PPOTrainer

ppo_trainer = PPOTrainer(
    args=ppo_config,

    processing_class=policy_tokenizer,

    model=policy_model,

    ref_model=ref_model,

    reward_model=ppo_reward_model,

    train_dataset=ppo_dataset,

    value_model=value_model
)

print("PPO Trainer created successfully!")

print("PPO Trainer created successfully!")

PPO Trainer created successfully!
PPO Trainer created successfully!


In [ ]:
                         PROMPT
                           │
                           ▼
                    ┌────────────┐
                    │   GPT-2    │
                    │   POLICY   │
                    └─────┬──────┘
                          │
                          ▼
                   Generated Response
                          │
                          ▼
                 ┌─────────────────┐
                 │  Reward Model   │
                 │   DistilBERT    │
                 └────────┬────────┘
                          │
                          ▼
                       REWARD
                          │
            ┌─────────────┼─────────────┐
            │             │             │
            ▼             ▼             ▼
       Policy GPT-2   Reference GPT-2  Value Model
            │             │             │
            └─────────────┼─────────────┘
                          ▼
                         PPO
                          │
                          ▼
                   Updated GPT-2

In [ ]:
policy_model     # GPT-2 — being optimized
ref_model        # Frozen GPT-2 — reference policy
ppo_reward_model # DistilBERT — gives reward
value_model      # GPT-2-based value estimator
ppo_dataset      # Prompts
ppo_config       # PPO hyperparameters
ppo_trainer      # PPO algorithm

In [68]:
# Training

print("Starting PPO training...")

ppo_trainer.train()

print("PPO training completed!")

Starting PPO training...
===training policy===


Step,Training Loss


PPO training completed!


In [ ]:
                 RLHF PIPELINE
                     │
                     ▼
              ┌─────────────┐
              │   GPT-2     │
              │ Base Policy │
              └──────┬──────┘
                     │
                     ▼
              Generate Response
                     │
                     ▼
          ┌─────────────────────┐
          │   Reward Model      │
          │     DistilBERT      │
          └──────────┬──────────┘
                     │
                     ▼
                  Reward
                     │
                     ▼
              ┌─────────────┐
              │     PPO     │
              │ Optimization│
              └──────┬──────┘
                     │
                     ▼
              Updated GPT-2

In [ ]:
# Checking 
             BEFORE PPO
                 │
                 ▼
          GPT-2 generates
                 │
                 ▼
          Reward Model
                 │
                 ▼
             Reward A


             AFTER PPO
                 │
                 ▼
       Updated GPT-2 generates
                 │
                 ▼
          Reward Model
                 │
                 ▼
             Reward B

In [69]:
# ORGIIANL 

from transformers import AutoModelForCausalLM

# Original GPT-2 before PPO
before_ppo_model = AutoModelForCausalLM.from_pretrained("gpt2")

before_ppo_model.eval()
policy_model.eval()

print("Before-PPO and After-PPO models are ready!")

Before-PPO and After-PPO models are ready!


In [74]:
# just creating a function
def generate_response(model, prompt, max_new_tokens=30):

    device = next(model.parameters()).device

    inputs = policy_tokenizer(
        prompt,
        return_tensors="pt"
    )

    # Move inputs to the same device as the model
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=policy_tokenizer.eos_token_id
        )

    response = policy_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [75]:
# just creating a reward fucniton

def calculate_reward(prompt, response):

    text = prompt + "\n" + response

    inputs = reward_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    with torch.no_grad():

        output = reward_model.base_model(
            **inputs
        )

    reward = output.logits.squeeze().item()

    return reward

In [76]:
# COMPARING

prompt = "What is machine learning?"

# BEFORE PPO
before_response = generate_response(
    before_ppo_model,
    prompt
)

# AFTER PPO
after_response = generate_response(
    policy_model,
    prompt
)

print("=" * 70)
print("PROMPT")
print(prompt)

print("\n" + "=" * 70)
print("BEFORE PPO")
print(before_response)

print("\n" + "=" * 70)
print("AFTER PPO")
print(after_response)

PROMPT
What is machine learning?

BEFORE PPO
What is machine learning? How does it work? How can it be used to create better user experiences?

What is a self-learning system? How does it work

AFTER PPO
What is machine learning?

Machine learning has been around for a while, but it has recently gotten a lot more mainstream acceptance. Machines are used to classify data, to


In [77]:
def calculate_reward(prompt, response):

    text = prompt + "\n" + response

    inputs = reward_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    device = next(reward_model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    reward_model.eval()

    with torch.no_grad():

        output = reward_model.base_model(
            **inputs
        )

    reward = output.logits.squeeze().item()

    return reward

In [78]:
before_reward = calculate_reward(
    prompt,
    before_response
)

after_reward = calculate_reward(
    prompt,
    after_response
)

print("\nREWARD COMPARISON")
print(f"Before PPO : {before_reward:.4f}")
print(f"After PPO  : {after_reward:.4f}")


REWARD COMPARISON
Before PPO : -0.8286
After PPO  : -0.1730


Since our reward model was trained so that higher reward = preferred response, the increase means:

$$ -0.1730 > -0.8286 $$

and therefore the post-PPO response received a substantially higher reward.

What happened?
	Before PPO	After PPO
Response	More conversational / question-oriented	More directly explanatory
Reward	-0.8286	-0.1730
Change	—	+0.6556

                    BEFORE PPO
                        │
                        ▼
              "How does it work?
               How can it be used..."
                        │
                        ▼
                 Reward = -0.8286
                        │
                        │
                       PPO
                        │
                        ▼
                    AFTER PPO
                        │
                        ▼
              "Machine learning has
               been around for a while..."
                        │
                        ▼
                 Reward = -0.1730

Reward increased by 0.6556.